In [ ]:
import pandas as pd
import numpy as np
import os
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


In [ ]:
sales_path = "../data/raw/sales_updated.csv"
customers_path = "../data/raw/customers_updated.csv"

sales = pd.read_csv(sales_path)
customers = pd.read_csv(customers_path)

print("Sales shape:", sales.shape)
print("Customers shape:", customers.shape)


In [ ]:
# Basic data-quality checks
print("Sales missing values:\n", sales.isna().sum())
print("\nCustomer missing values:\n", customers.isna().sum())
print("\nDuplicate sales rows:", sales.duplicated().sum())
print("Duplicate transaction IDs:", sales["Transaction_ID"].duplicated().sum())
print("Duplicate customer IDs:", customers["Customer_ID"].duplicated().sum())


In [ ]:
# Recalculate customer spending from transaction data
customer_sales = (sales.groupby("Customer_ID", as_index=False)["Total_Amount"].sum()
                  .rename(columns={"Total_Amount":"Calculated_Spent"}))

customers_clean = customers.drop(columns=["Total_Spent"]).copy()
customers_clean = customers_clean.merge(customer_sales, on="Customer_ID", how="left")
customers_clean = customers_clean.rename(columns={"Calculated_Spent":"Total_Spent"})

validation = customers[["Customer_ID","Total_Spent"]].merge(
    customers_clean[["Customer_ID","Total_Spent"]], on="Customer_ID", suffixes=("_Raw","_Calculated")
)
validation["Difference"] = validation["Total_Spent_Raw"] - validation["Total_Spent_Calculated"]
print("Customers checked:", len(validation))
print("Matching corrected values:", (validation["Difference"].abs() == 0).sum())
print("Mismatching values:", (validation["Difference"].abs() > 0).sum())


In [ ]:
# Validate sales amount
sales["Calculated_Amount"] = sales["Quantity"] * sales["Unit_Price"]
sales["Amount_Difference"] = sales["Total_Amount"] - sales["Calculated_Amount"]
print("Sales checked:", len(sales))
print("Mismatches:", (sales["Amount_Difference"].abs() > 0.01).sum())

sales_clean = sales.drop(columns=["Calculated_Amount","Amount_Difference"]).copy()


In [ ]:
# Date transformation
sales_clean["Date"] = pd.to_datetime(sales_clean["Date"], errors="coerce")
sales_clean["Sale_Date"] = sales_clean["Date"].dt.date
sales_clean["Year"] = sales_clean["Date"].dt.year
sales_clean["Month"] = sales_clean["Date"].dt.month
sales_clean["Month_Name"] = sales_clean["Date"].dt.strftime("%B")
sales_clean["Year_Month"] = sales_clean["Date"].dt.to_period("M").astype(str)


In [ ]:
# Overall KPIs
overall_kpis = pd.DataFrame({
    "Metric":["Total Revenue","Total Quantity","Total Transactions","Average Transaction Value","Unique Customers","Unique Products"],
    "Value":[sales_clean["Total_Amount"].sum(), sales_clean["Quantity"].sum(), sales_clean["Transaction_ID"].nunique(), sales_clean["Total_Amount"].mean(), sales_clean["Customer_ID"].nunique(), sales_clean["Product_Name"].nunique()]
})
display(overall_kpis)


In [ ]:
# Monthly, category and product analysis
monthly_sales = sales_clean.groupby("Year_Month", as_index=False).agg(
    Transactions=("Transaction_ID","nunique"), Quantity=("Quantity","sum"), Revenue=("Total_Amount","sum")
)
category_sales = sales_clean.groupby("Product_Category", as_index=False).agg(
    Transactions=("Transaction_ID","nunique"), Quantity=("Quantity","sum"), Revenue=("Total_Amount","sum")
).sort_values("Revenue", ascending=False)
product_sales = sales_clean.groupby("Product_Name", as_index=False).agg(
    Transactions=("Transaction_ID","nunique"), Quantity=("Quantity","sum"), Revenue=("Total_Amount","sum")
).sort_values("Revenue", ascending=False)
display(monthly_sales)
display(category_sales)
display(product_sales)


In [ ]:
# Customer segmentation using revenue quartiles
q25 = customers_clean["Total_Spent"].quantile(0.25)
q75 = customers_clean["Total_Spent"].quantile(0.75)
customers_clean["Customer_Segment"] = pd.cut(
    customers_clean["Total_Spent"], bins=[-np.inf,q25,q75,np.inf],
    labels=["Low Value","Medium Value","High Value"], include_lowest=True
)
segment_summary = customers_clean.groupby("Customer_Segment", observed=False).agg(
    Customers=("Customer_ID","count"), Total_Revenue=("Total_Spent","sum"), Average_Revenue=("Total_Spent","mean")
).reset_index()
display(segment_summary)


In [ ]:
# Linear-regression revenue forecast for the next 3 months
from sklearn.linear_model import LinearRegression

forecast_monthly = monthly_sales[["Year_Month","Revenue"]].copy()
forecast_monthly["Month_Index"] = range(len(forecast_monthly))
model = LinearRegression()
model.fit(forecast_monthly[["Month_Index"]], forecast_monthly["Revenue"])

future_months = pd.DataFrame({"Month_Index":[4,5,6],"Year_Month":["2024-05","2024-06","2024-07"]})
future_months["Predicted_Revenue"] = model.predict(future_months[["Month_Index"]]).round(2)
display(future_months)


In [ ]:
# Customer Lifetime Value (CLV), using the project/reference formula
customer_clv = (sales_clean.groupby("Customer_ID")
    .agg(Total_Spent=("Total_Amount","sum"), Purchase_Frequency=("Transaction_ID","nunique"))
    .reset_index())
customer_clv["Average_Purchase_Value"] = customer_clv["Total_Spent"] / customer_clv["Purchase_Frequency"]
customer_lifespan = 3
customer_clv["CLV"] = customer_clv["Average_Purchase_Value"] * customer_clv["Purchase_Frequency"] * customer_lifespan
customer_clv = customer_clv.merge(customers_clean[["Customer_ID","Customer_Segment"]], on="Customer_ID", how="left")
customer_clv = customer_clv.sort_values("CLV", ascending=False).reset_index(drop=True)
display(customer_clv.head(10))


In [ ]:
# Prepare forecast export and processed datasets
monthly_forecast = monthly_sales[["Year_Month","Revenue"]].rename(columns={"Revenue":"Actual_Revenue"}).copy()
monthly_forecast = pd.concat([
    monthly_forecast,
    future_months.rename(columns={"Predicted_Revenue":"Actual_Revenue"})[["Year_Month","Actual_Revenue"]].assign(Actual_Revenue=np.nan)
]).merge(future_months, on="Year_Month", how="left")
monthly_forecast["Predicted_Revenue"] = monthly_forecast["Predicted_Revenue"].fillna(np.nan)

os.makedirs("../data/processed", exist_ok=True)
sales_clean.to_csv("../data/processed/sales_processed.csv", index=False)
customers_clean.to_csv("../data/processed/customers_processed.csv", index=False)
customer_clv.to_csv("../data/processed/customer_clv.csv", index=False)
monthly_forecast.to_csv("../data/processed/monthly_forecast.csv", index=False)
print("Processed datasets exported successfully.")
